In [1]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

PERIOD_BINS = [1800, 1920, 1940, 1960, 1990, 2030]
PERIOD_LABELS = ["before1920", "1920-39", "1940-59", "1960-89", "1990+"]
feature_names = ["high-rise", "garage", "mixed-use", "service", "residential"]
EXC_FUNC_FOR_SERVICE = [
    "garaże",
    "mieszkaniowa",
    "pustostan",
    "brak parteru",
    "przemysł i infrastruktura",
]


def feature_types(d, high_rise):
    return [
        d["number_of_storeys"].ge(high_rise),
        d["is_garage"],
        d["is_service"] & d["upper_floor_function"].str.contains("mieszkaniowa"),
        d["is_service"],
        d["is_residential"] | d["upper_floor_function"].str.contains("mieszkaniowa"),
    ]


def add_building_features(
    df_short, df_long, high_rise=10, bins=PERIOD_BINS, labels=PERIOD_LABELS
):

    data_long_features = (
        df_long.assign(
            is_garage=lambda d: d["category"] == "garaże",
            is_residential=lambda d: d["category"] == "mieszkaniowa",
            is_vacant=lambda d: d["category"] == "pustostan",
            is_service=lambda d: ~d["category"].isin(EXC_FUNC_FOR_SERVICE),
        )
        .groupby("id")
        .any()
        .loc[:, ["is_garage", "is_residential", "is_vacant", "is_service"]]
    )

    return df_short.join(data_long_features, validate="one_to_one").assign(
        type=lambda d: np.select(feature_types(d, high_rise), feature_names, "other"),
        time_period=lambda d: pd.cut(
            d["build_year_est"], bins, right=False, labels=labels
        ),
    )


def street_metrics(
    df_long, df_short=None, min_functional=10, cat_omitted=("brak parteru", "pustostan")
):

    streets_long = pd.concat(
        [
            df_long.loc[lambda d: d["category"] != "brak parteru"]
            .assign(is_vacant=lambda d: d["category"] == "pustostan")
            .groupby("addr_street")
            .agg(num_premises=("category", "size"), num_vacant=("is_vacant", "sum"))
            .assign(
                num_premises_wo_vacant=lambda d: d["num_premises"] - d["num_vacant"],
                share_vacant=lambda d: d["num_vacant"].div(d["num_premises"]),
            ),
            df_long.loc[lambda d: ~d["category"].isin(cat_omitted)]
            .groupby("addr_street")["category"]
            .value_counts()
            .to_frame("premise_per_cat")
            .assign(
                total_premises_per_street=lambda d: d.groupby("addr_street")[
                    "premise_per_cat"
                ].transform("sum"),
                share=lambda d: d["premise_per_cat"].div(
                    d["total_premises_per_street"]
                ),
                lnshare_mult_share=lambda d: np.log(d["share"]).mul(d["share"]).mul(-1),
            )
            .groupby("addr_street")
            .agg(
                num_cat=("premise_per_cat", lambda s: s.gt(0).sum()),
                leader_share=("share", "max"),
                func_entropy=("lnshare_mult_share", "sum"),
            )
            .assign(eff_cat=lambda d: np.exp(d["func_entropy"])),
        ],
        axis=1,
    ).loc[
        :,
        [
            "num_premises",
            "num_premises_wo_vacant",
            "num_vacant",
            "share_vacant",
            "leader_share",
            "func_entropy",
            "num_cat",
            "eff_cat",
        ],
    ]
    if df_short is not None:
        streets_long = streets_long.join(
            df_short.groupby("addr_street")
            .agg(
                num_buildings=("osm_id", "size"),
                num_mix_use=("type", lambda d: (d == "mixed-use").sum()),
            )
            .assign(share_mix_use=lambda d: d["num_mix_use"].div(d["num_buildings"]))
        )
    return streets_long.loc[lambda d: d["num_premises_wo_vacant"].ge(min_functional)]


In [2]:
data_gdynia = pd.read_csv(
    "../../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_long_categories = pd.read_csv(
    "../../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

1.

Num of all share how much of all premises are in mixed-use buildings

In [3]:
def ground_floor_profile(df_long, df_short, types=("mixed-use", "service")):

    premises_wth_types = df_long.join(df_short.loc[:, "type"], validate="many_to_one")

    return (
        pd.crosstab(
            index=premises_wth_types.reset_index()["category"],
            columns=premises_wth_types.reset_index().loc[
                lambda d: d["type"].isin(types), "type"
            ],
            normalize="columns",
        )
        .assign(
            div_1_2=lambda d: d.loc[:, types[0]].div(d.loc[:, types[1]]),
            sum_build_types=pd.crosstab(
                index=premises_wth_types.reset_index()["category"],
                columns=premises_wth_types.reset_index().loc[
                    lambda d: d["type"].isin(types), "type"
                ],
            ).sum(axis=1),
            is_sum_gt_10=lambda d: d["sum_build_types"].ge(10),
        )
        .sort_values("div_1_2", ascending=False)
    )

In [4]:
data_gdynia_features = add_building_features(data_gdynia, data_gdynia_long_categories)
ground_floor_profile(data_gdynia_long_categories, data_gdynia_features)

type,mixed-use,service,div_1_2,sum_build_types,is_sum_gt_10
category,,,,,
mieszkaniowa,0.004910,0.000000,inf,3,False
pustostan,0.029460,0.003067,9.603928,19,True
usługi,0.193126,0.113497,1.701597,155,True
handel,0.346972,0.230061,1.508172,287,True
finanse,0.054010,0.036810,1.467267,45,True
zdrowie,0.109656,0.082822,1.323998,94,True
gastronomia,0.206219,0.159509,1.292836,178,True
brak parteru,0.013093,0.018405,0.711402,14,True
kultura i rozrywka,0.018003,0.070552,0.255177,34,True


In [5]:
ground_floor_profile(data_gdynia_long_categories, data_gdynia_features).iloc[
    :, [0, 1]
].style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

type,mixed-use,service
category,,
mieszkaniowa,0.5%,
pustostan,2.9%,0.3%
usługi,19.3%,11.3%
handel,34.7%,23.0%
finanse,5.4%,3.7%
zdrowie,11.0%,8.3%
gastronomia,20.6%,16.0%
brak parteru,1.3%,1.8%
kultura i rozrywka,1.8%,7.1%


2.

In [6]:
pd.crosstab(
    index=data_gdynia_features["condition"],
    columns=data_gdynia_features["is_vacant"],
    normalize="index",
)

is_vacant,False,True
condition,,
1,0.628571,0.371429
2,0.966408,0.033592
3,0.947598,0.052402


In [7]:
def vacancy_kind_cond(d):
    perc_vacacy = d.iloc[:, 1].div(d["sum_premises_per_build"])
    return [perc_vacacy == 0, perc_vacacy == 1.0]


vacancy_kind_types = ["without vacant", "full gf vacant"]


def add_vacacy_kind(df_short, df_long):

    df_long = df_long.loc[lambda d: d["category"] != "brak parteru"]

    return df_short.join(
        pd.crosstab(
            df_long.reset_index()["id"],
            df_long.reset_index()["category"].eq("pustostan"),
        )
        .assign(sum_premises_per_build=lambda d: d.sum(axis=1))
        .assign(
            vacancy_kind=lambda d: np.select(
                vacancy_kind_cond(d), vacancy_kind_types, "vacant among working"
            )
        )
        .fillna({"vacacy_kind": "without vacant"})
    )


In [8]:
data_gdynia_vacant = add_vacacy_kind(data_gdynia_features, data_gdynia_long_categories)
pd.crosstab(
    index=data_gdynia_vacant.reset_index()["condition"],
    columns=data_gdynia_vacant.reset_index()["vacancy_kind"],
    normalize="index",
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

vacancy_kind,full gf vacant,vacant among working,without vacant
condition,,,
1,31.4%,5.7%,62.9%
2,1.9%,1.4%,96.6%
3,2.7%,2.7%,94.7%


In [9]:
pd.crosstab(
    index=data_gdynia_vacant.reset_index()["vacancy_kind"],
    columns=[data_gdynia_vacant.reset_index()["time_period"]],
    normalize="columns",
).style.background_gradient(cmap="Blues", axis="index").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

time_period,before1920,1920-39,1940-59,1960-89,1990+
vacancy_kind,,,,,
full gf vacant,,3.7%,3.1%,2.9%,2.6%
vacant among working,,6.1%,0.7%,,1.7%
without vacant,100.0%,90.2%,96.2%,97.1%,95.7%


In [10]:
pd.crosstab(
    index=data_gdynia_vacant.reset_index()["addr_street"],
    columns=data_gdynia_vacant.reset_index()["vacancy_kind"].eq("vacant among working"),
    normalize="index",
).rename(columns={False: "no vacant among working", True: "vacant among working"}).join(
    pd.crosstab(
        index=data_gdynia_vacant.reset_index()["addr_street"],
        columns=data_gdynia_vacant.reset_index()["vacancy_kind"].eq(
            "vacant among working"
        ),
        margins=True,
    ).rename(columns={"All": "sum_build_per_street"})["sum_build_per_street"]
).loc[lambda d: d["vacant among working"].gt(0)].style.background_gradient(
    cmap="Blues", axis="index", subset=["vacant among working"]
).format(
    lambda v: "" if v == 0 else f"{v:.1%}",
    subset=["no vacant among working", "vacant among working"],
)

,no vacant among working,vacant among working,sum_build_per_street
addr_street,,,
Antoniego Abrahama,97.1%,2.9%,70
Arkadiusza Rybickiego,77.8%,22.2%,9
Generała Józefa Bema,95.7%,4.3%,23
Skwer Tadeusza Kościuszki,91.7%,8.3%,12
Starowiejska,98.1%,1.9%,53
Świętojańska,88.7%,11.3%,106


3.

In [11]:
def building_type_shares(df_short):

    return pd.crosstab(
        index=df_short.reset_index()["addr_street"],
        columns=df_short.reset_index()["type"],
        normalize="index",
    )


def entropy_from_shares(df_share):
    df_share = df_share.replace(0, np.nan)
    return (
        (np.log(df_share) * df_share)
        .sum(axis=1)
        .mul(-1)
        .to_frame(name="type_entropy")
        .assign(eff_types=lambda d: np.exp(d["type_entropy"]))
    )

In [12]:
streets_entropy = street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).join(entropy_from_shares(building_type_shares(data_gdynia_features)))
streets_entropy.loc[:, ["type_entropy", "func_entropy", "num_premises_wo_vacant"]].corr(
    method="spearman"
)

,type_entropy,func_entropy,num_premises_wo_vacant
type_entropy,1.000000,0.562671,-0.004110
func_entropy,0.562671,1.000000,0.304932
num_premises_wo_vacant,-0.004110,0.304932,1.000000


In [13]:
building_type_shares(data_gdynia_features).join(
    entropy_from_shares(building_type_shares(data_gdynia_features))["type_entropy"]
).loc[streets_entropy.index].sort_values(
    "type_entropy", ascending=False
).style.background_gradient(
    cmap="Blues",
    axis=None,
    subset=["garage", "high-rise", "mixed-use", "other", "residential", "service"],
).format(
    lambda v: "" if v == 0 else f"{v:.1%}",
    subset=["garage", "high-rise", "mixed-use", "other", "residential", "service"],
).format(lambda v: f"{v:.3f}", subset="type_entropy")

,garage,high-rise,mixed-use,other,residential,service,type_entropy
addr_street,,,,,,,
3 Maja,9.5%,9.5%,42.9%,,14.3%,23.8%,1.431
Jerzego Waszyngtona,4.2%,16.7%,4.2%,4.2%,33.3%,37.5%,1.430
Armii Krajowej,4.0%,4.0%,36.0%,,24.0%,32.0%,1.332
Wójta Radtkego,,15.4%,25.6%,,17.9%,41.0%,1.311
Żołnierzy I Armii Wojska Polskiego,,14.3%,42.9%,,14.3%,28.6%,1.277
Świętego Piotra,,10.0%,20.0%,,20.0%,50.0%,1.221
Stefana Batorego,,,50.0%,12.5%,12.5%,25.0%,1.213
Władysława IV,,16.2%,45.9%,5.4%,,32.4%,1.175
Kazimierza Pułaskiego,38.5%,,,,30.8%,30.8%,1.093


In [15]:
streets_entropy.sort_values("func_entropy", ascending=False).iloc[2:].loc[
    :, ["func_entropy", "type_entropy"]
].corr(method="spearman")

,func_entropy,type_entropy
func_entropy,1.000000,0.460513
type_entropy,0.460513,1.000000


In [16]:
assert np.isclose(
    street_metrics(data_gdynia_long_categories).loc["Świętojańska", "func_entropy"],
    1.523,
    atol=0.001,
)
assert len(street_metrics(data_gdynia_long_categories)) == 28

In [17]:
street_metrics(
    data_gdynia_long_categories,
    cat_omitted=EXC_FUNC_FOR_SERVICE,
).join(
    entropy_from_shares(building_type_shares(data_gdynia_features))["type_entropy"]
).loc[:, ["func_entropy", "type_entropy"]].corr(method="spearman")

,func_entropy,type_entropy
func_entropy,1.000000,0.365627
type_entropy,0.365627,1.000000
